## Agent Tools

### Warum brauchen Agents Tools?

#### Das Problem:

Ohne Tools ist das Wissen vom Agenten wie eingefroren.
Er weiß nur das, was er beim Training gelernt hat – also keine aktuellen News, keine Infos aus deinem Unternehmen, nix Neues.
Außerdem hat er keine Verbindung zur echten Welt.
Das heißt:

- Er kann nichts nachschauen
- Er kann nichts ausführen
- Er kann dir eigentlich nichts „erledigen“


#### Die Lösung:

Tools machen aus einem isolierten Sprachmodell einen richtig nützlichen Agenten.

 Ganz einfach gesagt:
Tools sind das, was den Agenten handlungsfähig macht.

Mit Tools kann er zum Beispiel:

- aktuelle Daten abrufen
- mit Systemen sprechen (APIs)
- Aktionen durchführen (z. B. E-Mails senden, Daten speichern)

In [1]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv, dotenv_values


for p in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if p.exists():
        load_dotenv(p,override=True)
        parsed = dotenv_values(str(p))
        if parsed.get("GOOGLE_API_KEY"):
            os.environ["GOOGLE_API_KEY"] = parsed["GOOGLE_API_KEY"].strip()
        break

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError(
        "GOOGLE_API_KEY fehlt. Lege .env mit GOOGLE_API_KEY=... an oder setze die Variable in der Shell."
    )

print("✅ Setup and authentication complete.")

✅ Setup and authentication complete.


In [3]:
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [4]:
retry_config = types.HttpRetryOptions(
    # Maximum retry attempts
    attempts=5,  
    # Delay multiplier
    exp_base=7,  
    initial_delay=1,
    # Retry on these HTTP errors
    http_status_codes=[429, 500, 503, 504],  
)

#### Was sind Custom Tools?
Custom Tools sind Werkzeuge, die wir selbst mit eigenem Code bauen. Sie sind nicht allgemein vorgefertigt, sondern genau an die Aufgabe oder dein Unternehmen angepasst.

Der große Vorteil ist: Wir haben volle Kontrolle darüber, was das Tool macht, welche Daten es nutzt und wie es reagiert.

#### Wann nutzt man sie?
Man nutzt Custom Tools immer dann, wenn Standard Tools nicht reichen. Allgemeine Tools wie Websuche oder Code Ausführung sind nützlich, aber jedes Unternehmen hat eigene Regeln, eigene Daten und eigene Prozesse.

Custom Tools helfen uns, diese speziellen Anforderungen direkt abzubilden und mit internen Systemen zu verbinden.

Das Beispiel mit dem Währungsrechner
Im Beispiel gibt es einen Agenten, der Geld von einer Währung in eine andere umrechnet. Dabei soll er nicht nur den Wechselkurs benutzen,sondern auch die Gebühren berücksichtigen.

Dafür nutzt der Agent zwei eigene Tools:

- Ein Tool sucht die Gebühr für den Wechsel heraus.

- Und ein anderes Tool liefert den Wechselkurs.

Danach macht der Agent den Rechenschritt und berechnet die gesamten Kosten inklusive Gebühren.

#### Warum das wichtig ist
Das Beispiel zeigt gut, wie Agenten arbeiten: Sie teilen eine Aufgabe in kleine Schritte auf, holen sich die nötigen Informationen über Tools und kombinieren dann alles zu einer fertigen Antwort.

So kann eine KI nicht nur allgemein antworten, sondern auch konkrete geschäftliche Aufgaben zuverlässig lösen.




#### Was bedeutet „Tool definieren“?
Ein Tool in ADK ist im Grunde einfach eine Python Funktion, die der Agent benutzen darf. Damit das klappt, schreiben wir eine normale Funktion, beschreiben sie sauber und fügen sie dann in die Tool Liste des Agenten ein.

Der Agent muss dann nicht selbst wissen, wie die Funktion intern arbeitet. ADK übernimmt die Verbindung und sorgt dafür, dass der Agent das Tool richtig aufrufen kann.

#### Die 3 wichtigsten Schritte
1. Wir erstellen eine Python Funktion.

2. Wir achten auf gute Beschreibung, Typen und Fehlerbehandlung

3. Wir tragen die Funktion in die tools=[]-Liste des Agenten ein.

Das ist der praktische Kern: Die Funktion wird gebaut, und ADK macht daraus ein nutzbares Agenten Tool.

#### Warum Docstrings so wichtig sind
Ein Docstring ist die kurze Textbeschreibung direkt in der Funktion. Er erklärt dem Modell, was das Tool macht, wann es benutzt werden soll und welche Eingaben erwartet werden.

Für das Modell ist das extrem wichtig, weil es nicht „von selbst“ weiß, was dein Code tut. Es liest die Beschreibung wie eine Anleitung.

#### Warum Type Hints wichtig sind
Type Hints sagen, welche Art von Daten erwartet wird, zum Beispiel str, int oder dict. Dadurch kann ADK automatisch ein passendes Schema erzeugen.

Das hilft dabei, Fehler zu vermeiden, weil der Agent dann besser versteht, welche Eingaben gültig sind.

#### Warum strukturierte Rückgaben empfohlen werden
Die Best Practice im Text sagt: Gib Ergebnisse am besten als Dictionary zurück, zum Beispiel:

- Erfolg: {"status": "success", "data": ...}

- Fehler: {"status": "error", "error_message": ...}

Das ist gut, weil der Agent dann sofort erkennt, ob etwas funktioniert hat oder nicht, und wie er weiter reagieren soll.

#### Warum Fehlerbehandlung so wichtig ist
Wenn ein Tool scheitert, sollte es nicht einfach „kaputtgehen“, sondern klar sagen, was schiefgelaufen ist. Das Modell kann mit einer guten Fehlermeldung oft besser umgehen als mit einem bloßen technischen Fehlercode.

So kann der Agent etwa entscheiden, ob er es erneut versucht, den Nutzer nachfragt oder einen anderen Weg nimmt.


In [5]:
def get_fee_for_payment_method(method: str) -> dict:
    """Looks up the transaction fee percentage for a given payment method.

    This tool simulates looking up a company's internal fee structure based on
    the name of the payment method provided by the user.

    Args:
        method: The name of the payment method. It should be descriptive,
                e.g., "platinum credit card" or "bank transfer".

    Returns:
        Dictionary with status and fee information.
        Success: {"status": "success", "fee_percentage": 0.02}
        Error: {"status": "error", "error_message": "Payment method not found"}
    """
    
    fee_database = {
        # 2%
        "platinum credit card": 0.02,  
        # 3.5%
        "gold debit card": 0.035,  
        # 1%
        "bank transfer": 0.01,  
    }

    fee= fee_database.get(method.lower())
    if fee is not None:
        return {"status": "success","fee_percentage": fee}
    else:
        return {
            "status": "error",
            "error_message": f"Payment method '{method}' not found",
        }


print("✅ Fee lookup function created")
print(f"Test: {get_fee_for_payment_method('platinum credit card')}")

✅ Fee lookup function created
Test: {'status': 'success', 'fee_percentage': 0.02}


In [6]:
def get_exchange_rate(base_currency: str, target_currency: str) -> dict:
    """Looks up and returns the exchange rate between two currencies.

    Args:
        base_currency: The ISO 4217 currency code of the currency you
                       are converting from (e.g., "USD").
        target_currency: The ISO 4217 currency code of the currency you
                         are converting to (e.g., "EUR").

    Returns:
        Dictionary with status and rate information.
        Success: {"status": "success", "rate": 0.93}
        Error: {"status": "error", "error_message": "Unsupported currency pair"}
    """

    # Static data simulating a live exchange rate API
    # In production, this would call something like: requests.get("api.exchangerates.com")
    rate_database = {
        "usd": {
             # Euro
            "eur": 0.93, 
            # Japanese Yen
            "jpy": 157.50,  
            # Indian Rupee
            "inr": 83.58,  
        }
    }

    # Input validation and processing
    base= base_currency.lower()
    target = target_currency.lower()

    # Return structured result with status
    rate =rate_database.get(base, {}).get(target)
    if rate is not None:
        return {"status": "success", "rate": rate}
    else:
        return {
            "status": "error",
            "error_message": f"Unsupported currency pair: {base_currency}/{target_currency}",
        }


print("Exchange rate function created")
print(f"💱 Test: {get_exchange_rate('USD', 'EUR')}")

Exchange rate function created
💱 Test: {'status': 'success', 'rate': 0.93}


Now let's create our currency agent. Pay attention to how the agent's instructions reference the tools:

Key Points:

The tools=[] list tells the agent which functions it can use
Instructions reference tools by their exact function names (e.g., 

get_fee_for_payment_method())
The agent uses these names to decide when and how to call each tool

In [9]:
# Currency agent with custom function tools
currency_agent = LlmAgent(
    name="currency_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a smart currency conversion assistant.

    For currency conversion requests:
    1. Use `get_fee_for_payment_method()` to find transaction fees
    2. Use `get_exchange_rate()` to get currency conversion rates
    3. Check the "status" field in each tool's response for errors
    4. Calculate the final amount after fees based on the output from `get_fee_for_payment_method` and `get_exchange_rate` methods and provide a clear breakdown.
    5. First, state the final converted amount.
        Then, explain how you got that result by showing the intermediate amounts. Your explanation must include: the fee percentage and its
        value in the original currency, the amount remaining after the fee, and the exchange rate used for the final conversion.

    If any tool returns status "error", explain the issue to the user clearly.
    """,
    tools=[get_fee_for_payment_method, get_exchange_rate],
)

print(" Currency agent created with custom function tools")
print(" Available tools:")
print("  • get_fee_for_payment_method for Looks up company fee structure")
print("  • get_exchange_rate for  Gets current exchange rates")

 Currency agent created with custom function tools
 Available tools:
  • get_fee_for_payment_method for Looks up company fee structure
  • get_exchange_rate for  Gets current exchange rates


In [10]:
# Test the currency agent
currency_runner = InMemoryRunner(agent=currency_agent)
_ = await currency_runner.run_debug(
    "I want to convert 500 US Dollars to Euros using my Platinum Credit Card. How much will I receive?"
)


 ### Created new session: debug_session_id

User > I want to convert 500 US Dollars to Euros using my Platinum Credit Card. How much will I receive?


_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 4.820005915s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate

Der Agent soll am Ende einen Endbetrag nach Gebühren berechnen. Das Problem ist: Sprachmodelle sind beim Rechnen nicht immer zuverlässig. Sie können sich verrechnen, Formeln falsch anwenden oder bei mehreren Schritten inkonsistent werden.

- Darum ist die Grundidee: Nicht das Modell selbst rechnen lassen, sondern den Rechenschritt an Code übergeben.

#### Warum ist das besser?
Code ist für Mathe viel verlässlicher als ein Sprachmodell, das nur „im Kopf“ rechnet.
Wenn der Agent also eine Berechnung braucht, kann er stattdessen Python Code erzeugen und ausführen lassen. Dann kommt das Ergebnis aus echter Berechnung, nicht aus geschätztem Sprachverständnis.

Das ist besonders nützlich bei Dingen wie:

Gebührenberechnung.

Währungsumrechnung.

Prozentrechnung.

Summen, Durchschnittswerte oder mehrstufige Formeln.

#### Was macht der Code Executor?
Der eingebaute Code Ausführer ist eine Umgebung, in der der Agent Code sicher ausführen kann.
Man kann sich das wie eine kleine geschützte Rechenumgebung vorstellen: Der Agent schreibt den Rechencode, die Umgebung führt ihn aus, und das Ergebnis wird zurückgegeben.

* Der wichtige Punkt ist: Der Agent denkt die Aufgabe, aber der Code macht die Rechnung.

#### Was ist der Vorteil für Agenten?
Dadurch werden Agenten zuverlässiger.
Statt dass das Modell eine Zahl „glaubt“, kann es die Zahl berechnen lassen. Das reduziert Fehler und macht Antworten konsistenter, besonders bei Aufgaben mit klaren mathematischen Regeln.


#### Der Kern in einem Satz
 Wenn ein Agent rechnen muss, soll er dafür lieber Code ausführen als selbst im Text zu schätzen.

In [11]:
calculation_agent = LlmAgent(
    name="CalculationAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a specialized calculator that ONLY responds with Python code. You are forbidden from providing any text, explanations, or conversational responses.
 
     Your task is to take a request for a calculation and translate it into a single block of Python code that calculates the answer.
     
     **RULES:**
    1.  Your output MUST be ONLY a Python code block.
    2.  Do NOT write any text before or after the code block.
    3.  The Python code MUST calculate the result.
    4.  The Python code MUST print the final result to stdout.
    5.  You are PROHIBITED from performing the calculation yourself. Your only job is to generate the code that will perform the calculation.
   
    Failure to follow these rules will result in an error.
       """,
    # Use the built-in Code Executor Tool. This gives the agent code execution capabilities   
    code_executor=BuiltInCodeExecutor(),  
)

#### Update the Agent's instruction and toolset

In [12]:
enhanced_currency_agent = LlmAgent(
    name="enhanced_currency_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    # Updated instruction
    instruction="""You are a smart currency conversion assistant. You must strictly follow these steps and use the available tools.

  For any currency conversion request:

   1. Get Transaction Fee: Use the get_fee_for_payment_method() tool to determine the transaction fee.
   2. Get Exchange Rate: Use the get_exchange_rate() tool to get the currency conversion rate.
   3. Error Check: After each tool call, you must check the "status" field in the response. If the status is "error", you must stop and clearly explain the issue to the user.
   4. Calculate Final Amount (CRITICAL): You are strictly prohibited from performing any arithmetic calculations yourself. You must use the calculation_agent tool to generate Python code that calculates the final converted amount. This 
      code will use the fee information from step 1 and the exchange rate from step 2.
   5. Provide Detailed Breakdown: In your summary, you must:
       * State the final converted amount.
       * Explain how the result was calculated, including:
           * The fee percentage and the fee amount in the original currency.
           * The amount remaining after deducting the fee.
           * The exchange rate applied.
    """,
    tools=[
        get_fee_for_payment_method,
        get_exchange_rate,
        AgentTool(agent=calculation_agent),  # Using another agent as a tool!
    ],
)

print("Enhanced currency agent created")
print(" New capability: Delegates calculations to specialist agent")
print(" Tool types used:")
print("  • Function Tools (fees, rates)")
print("  • Agent Tool (calculation specialist)")

Enhanced currency agent created
 New capability: Delegates calculations to specialist agent
 Tool types used:
  • Function Tools (fees, rates)
  • Agent Tool (calculation specialist)


In [13]:
enhanced_runner = InMemoryRunner(agent=enhanced_currency_agent)

In [ ]:
# Test the enhanced agent
response = await enhanced_runner.run_debug(
    "Convert 1,250 USD to INR using a Bank Transfer. Show me the precise calculation."
)


 ### Created new session: debug_session_id

User > Convert 1,250 USD to INR using a Bank Transfer. Show me the precise calculation.
